In [16]:
import pandas as pd 
import numpy as np 
import warnings 
warnings.filterwarnings('ignore')
np.random.seed(42)

df = pd.read_csv("AAPL_data.csv", parse_dates = ['Date'], index_col = 'Date').dropna()
df['Return'] = df['Close'].pct_change()
df['LogReturn'] = np.log(df['Close']).diff()
df['Target_Next'] = df['Close'].shift(-1)
df['Dir_Next'] = (df['Target_Next'] > df['Close']).astype(int)

df = df.dropna()

FEAT = ["Open","High","Low","Volume","RSI","MACD","MACD_signal","MACD_diff",
        "MA_20","STD_20","Upper_Band","Lower_Band"]
print(df.shape)


(2477, 19)


In [2]:
df.head()

,Open,High,Low,Close,Volume,Dividends,Stock Splits,MA_20,STD_20,Upper_Band,Lower_Band,RSI,MACD,MACD_signal,MACD_diff,Return,LogReturn,Target_Next,Dir_Next
Date,,,,,,,,,,,,,,,,,,,
2016-10-27 00:00:00-04:00,26.388035,26.495517,26.093030,26.179932,138248000,0.0,0.0,26.477680,0.435328,27.348336,25.607024,48.145836,0.298585,0.402426,-0.103842,NaN,NaN,26.006128,0
2016-10-28 00:00:00-04:00,26.040432,26.346869,25.944382,26.006128,151446800,0.0,0.0,26.485341,0.424980,27.335301,25.635381,44.733489,0.230078,0.367957,-0.137878,-0.006639,-0.006661,25.964970,0
2016-10-31 00:00:00-04:00,25.990125,26.122763,25.887216,25.964970,105677600,0.0,0.0,26.497004,0.405985,27.308973,25.685035,43.939279,0.170500,0.328465,-0.157965,-0.001583,-0.001584,25.496162,0
2016-11-01 00:00:00-04:00,25.946673,26.017565,25.276624,25.496162,175303200,0.0,0.0,26.479738,0.441150,27.362039,25.597438,36.081428,0.084481,0.279669,-0.195187,-0.018055,-0.018220,25.519033,1
2016-11-02 00:00:00-04:00,25.475584,25.692835,25.436708,25.519033,113326800,0.0,0.0,26.463044,0.471399,27.405842,25.520246,36.676394,0.017949,0.227325,-0.209376,0.000897,0.000897,25.245499,0


### Feature Engineering

In [ ]:
# Question 1: How do you scale OHLCV features?
from sklearn.preprocessing import StandardScaler

sc = StandardScaler().fit(df[['Open', 'High', 'Low', 'Close', 'Volume']])

print(sc.transform(df[['Open', 'High', 'Low', 'Close', 'Volume']]).std(axis = 0).round(2))

[1. 1. 1. 1. 1.]


In [ ]:
# Question 2: How do you normalize RSI into [0,1]?
from sklearn.preprocessing import MinMaxScaler

mm = MinMaxScaler().fit(df[['RSI']])
df['RSI_n'] = mm.transform(df[['RSI']])

print(df['RSI_n'].describe().round(3))

count    2479.000
mean        0.519
std         0.179
min         0.000
25%         0.381
50%         0.532
75%         0.649
max         1.000
Name: RSI_n, dtype: float64


In [ ]:
# Question 3: How do you build a FunctionTransformer for log returns?
from sklearn.preprocessing import FunctionTransformer

log_ret = FunctionTransformer(lambda x: np.log(x).diff().fillna(0), validate = False)
df['LogRet_ft'] = log_ret.fit_transform(df[['Close']])

print(df['LogRet_ft'].head())

Date
2016-10-27 00:00:00-04:00    0.000000
2016-10-28 00:00:00-04:00   -0.006661
2016-10-31 00:00:00-04:00   -0.001584
2016-11-01 00:00:00-04:00   -0.018220
2016-11-02 00:00:00-04:00    0.000897
Name: LogRet_ft, dtype: float64


In [ ]:
# Question 4: How do you create lag features using sklearn's FunctionTransformer?

from sklearn.preprocessing import FunctionTransformer

lag1 = FunctionTransformer(lambda x: pd.DataFrame(x).shift(1).values, validate = False)
lag5 = FunctionTransformer(lambda x: pd.DataFrame(x).shift(5).values, validate = False)

df['Close_lag1'] = lag1.fit_transform(df[['Close']]).ravel()
df['Close_lag5'] = lag5.fit_transform(df[['Close']]).ravel()

print(df[['Close', 'Close_lag1', 'Close_lag5']].tail())

                                Close  Close_lag1  Close_lag5
Date                                                         
2026-09-02 00:00:00-04:00  324.959991  325.130005  313.450012
2026-09-03 00:00:00-04:00  328.209991  324.959991  314.579987
2026-09-04 00:00:00-04:00  319.970001  328.209991  319.700012
2026-09-08 00:00:00-04:00  316.220001  319.970001  316.850006
2026-09-09 00:00:00-04:00  315.339996  316.220001  325.130005


In [ ]:
# Question 5: How do you build a PolynomialFeatures transformer for RSI/MACD?

from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree = 2, include_bias=False)
Xp = poly.fit_transform(df[['RSI', 'MACD_diff']])

print(poly.get_feature_names_out(['RSI', 'MACD_diff']))

['RSI' 'MACD_diff' 'RSI^2' 'RSI MACD_diff' 'MACD_diff^2']


In [ ]:
# Question 6: How do you bin volume into quantiles using KBinsDiscretizer?

from sklearn.preprocessing import KBinsDiscretizer

kb = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy = 'quantile')
df['Vol_bin'] = kb.fit_transform(df[['Volume']]).astype(int)

print(df['Vol_bin'].value_counts().sort_index())

Vol_bin
0    496
1    496
2    495
3    496
4    496
Name: count, dtype: int64


In [ ]:
# Question 7: How do you binarize RSI into overbought/ oversold via Binarizer?

from sklearn.preprocessing import Binarizer

ob = Binarizer(threshold=70).fit(df[['RSI']])
os_ = Binarizer(threshold=30).fit(df[['RSI']])
df['Overbought'] = ob.transform(df[['RSI']])
df['Oversold'] = 1 - os_.transform(df[['RSI']])

print(df[['Overbought', 'Oversold']].sum())

Overbought    351.0
Oversold       49.0
dtype: float64


In [13]:
# Question 8: How do you build a Pipeline for feature engineering?

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree = 2, include_bias=False)),
    ('sc', StandardScaler())
]).fit(df[['RSI', 'MACD_diff']])

print(pipeline.transform(df[['RSI', 'MACD_diff']]).shape)

(2479, 5)


In [14]:
# Question 9: How do you apply different transformers to OHLC vs Volume with ColumnTransformer

from sklearn.compose import ColumnTransformer

ct = ColumnTransformer([
    ('prices', StandardScaler(), ['Open', 'High', 'Low', 'Close']),
    ('vol', MinMaxScaler(), ['Volume']),
    ('rsi', StandardScaler(), ['RSI'])
])

print(ct.fit_transform(df).shape)

(2479, 6)


In [17]:
# Question 10: How do you select the top k stock feature with selectKBest?

from sklearn.feature_selection import SelectKBest, f_regression

skb = SelectKBest(f_regression, k = 6).fit(df[FEAT], df['Target_Next'])

print(df[FEAT].columns[skb.get_support()].to_list())

['Open', 'High', 'Low', 'MA_20', 'Upper_Band', 'Lower_Band']
